# Stacking — Ensemble via Meta-Learner

## What is Stacking?
Stacking (Stacked Generalisation) trains a **meta-learner** on top of multiple base models.

```
Training data
      │
      ├─── Base Model 1 (Random Forest)  ──→ predictions
      ├─── Base Model 2 (KNN)            ──→ predictions  ──→ Meta-Learner (LR) ──→ final prediction
      └─── Base Model 3 (Decision Tree)  ──→ predictions
```

### Stacking vs Voting
| | Voting | Stacking |
|---|---|---|
| How combined | Average / majority vote | Meta-learner trained on base predictions |
| Flexibility | Fixed combination rule | Learned combination |
| Risk | None | Meta-learner can overfit if not cross-validated |

### Stacking vs Bagging/Boosting
- **Bagging/Boosting** use the same algorithm with different data
- **Stacking** uses **different algorithms** and learns how to combine them


## Step 1: Imports and Load Dataset
Heart Disease dataset — binary classification (303 patients, 13 features).

In [141]:
import numpy as np
import pandas as pd

In [142]:
df = pd.read_csv('heart.csv')

In [143]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## Step 2: Prepare Features and Target

In [125]:
X = df.drop(columns=['target'])
y = df['target']

In [126]:
X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3


In [127]:
y

0      1
1      1
2      1
3      1
4      1
      ..
298    0
299    0
300    0
301    0
302    0
Name: target, Length: 303, dtype: int64

## Step 3: Train/Test Split

In [128]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=8)

In [129]:
print(X_train.shape)

(242, 13)


## Step 4: Define Base Estimators (Level-0 Models)
Three diverse classifiers — each learns a different representation of the data:
- `RandomForestClassifier` — ensemble of trees, captures non-linear patterns
- `KNeighborsClassifier` — distance-based, no assumptions about distribution
- `DecisionTreeClassifier` — single tree, fast and interpretable


In [130]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

In [136]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=10)),
    ('gbdt',GradientBoostingClassifier())
]

## Step 5: Define StackingClassifier
`final_estimator=LogisticRegression()` — the **meta-learner** (Level-1 model).

sklearn's `StackingClassifier` automatically:
1. Trains each base model using cross-validation on the training set
2. Uses their out-of-fold predictions as features for the meta-learner
3. Trains the meta-learner on those predictions
4. At test time: passes test data through all base models, then through the meta-learner


In [137]:
from sklearn.ensemble import StackingClassifier

clf = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression(),
    cv=10
)

## Step 6: Fit the Stacking Model

In [138]:
clf.fit(X_train, y_train)

StackingClassifier(cv=10,
                   estimators=[('rf',
                                RandomForestClassifier(n_estimators=10,
                                                       random_state=42)),
                               ('knn', KNeighborsClassifier(n_neighbors=10)),
                               ('gbdt', GradientBoostingClassifier())],
                   final_estimator=LogisticRegression())

## Step 7: Predict and Evaluate

In [139]:
y_pred = clf.predict(X_test)

In [140]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.8688524590163934

---
## Summary

```
StackingClassifier workflow:
  Train:  X_train → [RF, KNN, DT] → OOF predictions → LogisticRegression (meta)
  Predict: X_test → [RF, KNN, DT] → predictions → LogisticRegression → final

Key insight: The meta-learner learns WHEN to trust each base model.
If RF is good on certain feature ranges and KNN on others,
LogisticRegression learns to weight them accordingly.
```
